# XGBoost for Time Series  *(optional ML extension)*

*Based on: Chen, T. & Guestrin, C., XGBoost: A Scalable Tree Boosting System, KDD '16*

---

> This is an **optional extension** to the core curriculum. Notebooks 1–5
> already give you a complete classical-forecasting skillset; this one
> shows you how the same problem looks through a modern tabular-ML lens,
> including a failure mode you will absolutely hit in real projects.


## Learning Objectives

At the end of this notebook, you should be able to:

- Engineer **lag features** and **calendar features** from a `pandas.Series`
  so a tabular learner like XGBoost can forecast a time series.
- Diagnose the **trend-extrapolation failure** of tree models: trees never
  predict outside the training range, no matter how you encode time.
- Apply the standard **hybrid pattern**: a parametric model handles the
  trend, XGBoost handles the residuals, to forecast a trending series.
- Generate a multi-step forecast by **iterating** the model forward,
  feeding earlier predictions back in as lags.
- Place the hybrid model on a **like-for-like scoreboard** against
  seasonal-naive and Holt-Winters and read the result honestly.


## Setup

We use **weekly atmospheric CO2 at Mauna Loa**: roughly 2,250 observations
spanning 1958–2001. The series has a textbook combination of a strong
upward trend (about 1.5 ppm / year) and a clean 52-week seasonal cycle,
exactly the kind of series where you might reach for XGBoost in practice.

`NB_FAST=1` keeps the XGBoost fit cheap (fewer trees) so CI stays under
the per-notebook budget.


In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LinearRegression
from statsmodels.datasets import co2
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from xgboost import XGBRegressor

sns.set_theme(style="whitegrid", context="notebook")

NB_FAST = os.environ.get("NB_FAST", "").lower() in {"1", "true", "yes"}
warnings.filterwarnings("ignore")


Load weekly CO2 and hold out the **last 52 weeks (one year)** as the
test set.


In [ ]:
co2_series = co2.load_pandas().data.squeeze().dropna()
co2_series.name = "co2_ppm"

HORIZON = 52
train, test = co2_series.iloc[:-HORIZON], co2_series.iloc[-HORIZON:]
print(f"total: {len(co2_series)} weeks ({co2_series.index.min().date()} .. {co2_series.index.max().date()})")
print(f"train: {len(train):>5d}  |  test: {len(test):>5d}")


## 6.1 Why use ML for a time series at all?

Classical models (ARIMA, Holt-Winters) are tailored to time series, they
encode trend, seasonality, and autocorrelation directly in their structure.
ML models like XGBoost know **nothing** about time: they see a table of
rows and columns and try to predict one column from the others.

To make XGBoost forecast a time series, we have to **turn the temporal
structure into columns**. The two universal techniques are:

- **Lag features** (last week's value, last month's value, last year's
  value) become explicit columns.
- **Calendar features** (week-of-year, day-of-week, year) become
  explicit columns.

With those columns in place, forecasting is "just" supervised regression.
The skill is choosing the right features and *understanding their limits*,
which is what most of this notebook is about.


## 6.2 Lag features

A **lag-$k$ feature** is the value of the series $k$ steps in the past.
For weekly data, three lags carry most of the information:

- `lag_1`: last week (short-term momentum)
- `lag_4`: roughly a month ago (medium-term)
- `lag_52`: same week one year ago (yearly seasonality)


In [ ]:
def add_lags(s, lags=(1, 4, 52)):
    """Return a DataFrame with the series and one column per lag."""
    out = pd.DataFrame({"y": s})
    for k in lags:
        out[f"lag_{k}"] = s.shift(k)
    return out

raw_lags = add_lags(co2_series)
raw_lags.head(3)


The first 52 rows have `NaN` for `lag_52` (there is no value one year
before the start of the series). We will drop those rows before training.


## 6.3 Calendar features

XGBoost cannot tell that week 30 is consistently near the yearly maximum
unless we *give* it that information as a column. For weekly data the
natural calendar features are:

- `week`: integer 1..53 (week-of-year, captures seasonality)
- `year`: integer (long-term trend)


In [ ]:
def add_calendar(df):
    df = df.copy()
    df["week"] = df.index.isocalendar().week.astype(int)
    df["year"] = df.index.year
    return df


## 6.4 The trend trap: why naive XGBoost fails on this series

Here is the gotcha that bites everyone who tries XGBoost on time series
for the first time. Look at the **range** of the train and test sets:


In [ ]:
print(f"train range: {train.min():.1f} .. {train.max():.1f} ppm")
print(f"test  range: {test.min():.1f} .. {test.max():.1f} ppm")
print(f"test points strictly above train max: {(test > train.max()).sum()} / {len(test)}")


**The test set contains values XGBoost has never seen in training.** Every
split in an XGBoost tree is of the form `if feature < threshold then … else
…`. For any test row whose features push it past the highest training
example, every tree falls into its rightmost leaf, and that leaf was
fitted on the *highest-y training rows*. The prediction therefore gets
**clamped to roughly the training maximum**, no matter how cleverly we
encode time.

Tree models simply **cannot extrapolate**. Adding `t` or `year` as
features doesn't help, they hit the same wall.

The standard fix is a **hybrid model**: let a parametric model that
*can* extrapolate (here, a linear regression on time) handle the trend,
then train XGBoost on the **residuals**, which are stationary by
construction.


## 6.5 The hybrid model

The hybrid model splits the work in two: a parametric model captures the trend, and XGBoost models the seasonal structure left in the residuals. We build it in two steps.

### Step 1: Fit a linear trend

A linear regression on `t` captures the long-term direction. Crucially,
it predicts the trend for *any* future `t`, in or out of the training range.


In [ ]:
t_train = np.arange(len(train)).reshape(-1, 1)
t_test = np.arange(len(train), len(train) + len(test)).reshape(-1, 1)

trend_model = LinearRegression().fit(t_train, train.values)
trend_train = pd.Series(trend_model.predict(t_train), index=train.index, name="trend")
trend_test = pd.Series(trend_model.predict(t_test), index=test.index, name="trend")

print(f"slope: {trend_model.coef_[0]:.4f} ppm / week  "
      f"(= {trend_model.coef_[0] * 52:.2f} ppm / year)")


Plot the trend on top of the raw series so you can see exactly what the
linear regression captures, and what it leaves behind for XGBoost to
explain.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
co2_series.plot(ax=ax, color="lightgrey", label="raw CO2")
trend_train.plot(ax=ax, color="steelblue", label="linear trend (train)")
trend_test.plot(ax=ax, color="crimson", linestyle="--", label="linear trend (test, extrapolated)")
ax.set_title("Weekly CO2: linear trend handles the long-term level")
ax.set_ylabel("ppm")
ax.legend()
plt.tight_layout()
plt.show()


### Step 2: XGBoost on the residuals

Subtract the trend from the raw series. The residual is stationary,
it has a stable mean (zero, by construction) and a clear seasonal pattern.
This is exactly the kind of series tree models handle well.


In [ ]:
resid_train = train - trend_train
resid_train.name = "residual"

fig, ax = plt.subplots(figsize=(10, 3.0))
resid_train.plot(ax=ax, color="darkgreen", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Training residual (raw - linear trend): stationary, clearly seasonal")
ax.set_ylabel("ppm")
plt.tight_layout()
plt.show()


Build lag + calendar features on the **residual** (not the raw series),
drop the rows with `NaN` lags, and fit XGBoost.


In [ ]:
FEATURES = ["lag_1", "lag_4", "lag_52", "week"]

feat = add_calendar(add_lags(resid_train)).dropna()
X_train_xgb, y_train_xgb = feat[FEATURES], feat["y"]

n_est = 50 if NB_FAST else 500
xgb = XGBRegressor(
    n_estimators=n_est, max_depth=4, learning_rate=0.05,
    subsample=0.9, objective="reg:squarederror", random_state=0,
)
xgb.fit(X_train_xgb, y_train_xgb)

importances = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances.round(3)


## 6.6 Multi-step forecast: iterating residuals, adding trend back

For each future week we need to (a) know the lagged residuals, and
(b) call XGBoost to predict the next residual, then (c) add the
extrapolated trend back to get a prediction in ppm.

For step (a), `lag_1` and `lag_4` involve recent predictions, so we
iterate: each predicted residual is appended to the history that the
next step lags from.


In [ ]:
def hybrid_forecast(xgb, trend_test, resid_history, future_index, features):
    """Iteratively predict residuals, then add the trend back."""
    h = pd.Series(resid_history.values, index=resid_history.index, dtype=float)
    preds_ppm = []
    for date in future_index:
        row = {
            "lag_1": h.iloc[-1],
            "lag_4": h.iloc[-4],
            "lag_52": h.iloc[-52],
            "week": int(pd.Timestamp(date).isocalendar().week),
        }
        resid_hat = float(xgb.predict(pd.DataFrame([row])[features])[0])
        h.loc[date] = resid_hat                          # append for next iteration
        preds_ppm.append(resid_hat + trend_test.loc[date])  # reconstruct in original units
    return pd.Series(preds_ppm, index=future_index, name="hybrid_xgb")


Run the iterative hybrid forecast over the holdout and overlay it on the
last few years of the raw series.


In [ ]:
fc_hybrid = hybrid_forecast(xgb, trend_test, resid_train, test.index, FEATURES)

fig, ax = plt.subplots(figsize=(10, 3.8))
train.iloc[-260:].plot(ax=ax, color="lightgrey", label="train (last 5 years)")
test.plot(ax=ax, color="black", linewidth=1.5, label="actual")
fc_hybrid.plot(ax=ax, label="Linear trend + XGBoost residuals", color="darkorange", linewidth=1.4)
ax.set_title("Hybrid forecast over the last year")
ax.set_ylabel("ppm")
ax.legend()
plt.tight_layout()
plt.show()


## 6.7 Like-for-like scoreboard

Fit two classical baselines on the same training set and score everyone
with the same metrics on the same test set.

(SARIMA with seasonal period $m = 52$ takes minutes to fit on 2,000+
weekly observations, so it is intentionally omitted from this notebook,
add it yourself as a stretch exercise if you want.)


In [ ]:
def seasonal_naive(train, test_index, m=52):
    tile = np.tile(train.iloc[-m:].values, int(np.ceil(len(test_index) / m)))[: len(test_index)]
    return pd.Series(tile, index=test_index, name="seasonal_naive")

# Holt-Winters with weekly seasonality.
hw_fit = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=52,
    initialization_method="estimated",
).fit()
fc_hw = pd.Series(hw_fit.forecast(HORIZON).values, index=test.index, name="holt_winters")
fc_sn = seasonal_naive(train, test.index)


Score every forecast with MAE, RMSE, and MAPE on the held-out test set.
We compute on `.values` so index alignment can never silently destroy
the comparison.


In [ ]:
def mae(y, p): return np.mean(np.abs(y.values - p.values))
def rmse(y, p): return np.sqrt(np.mean((y.values - p.values) ** 2))
def mape(y, p): return np.mean(np.abs((y.values - p.values) / y.values)) * 100

forecasts = {"seasonal_naive": fc_sn, "holt_winters": fc_hw, "hybrid_xgb": fc_hybrid}

scoreboard = pd.DataFrame(
    {name: {"MAE": mae(test, fc), "RMSE": rmse(test, fc), "MAPE": mape(test, fc)}
     for name, fc in forecasts.items()},
).T.sort_values("RMSE")

scoreboard.round(3)


A few honest observations from the scoreboard:

- The **hybrid model** (linear trend + XGBoost residuals) is competitive
  with classical baselines, *because we separated the trend modelling
  from the seasonal/residual modelling*. The naive "XGBoost on raw y"
  approach we diagnosed in section 6.4 would score an order of magnitude
  worse on this data.
- **Seasonal-naive** is genuinely hard to beat on a series with strong
  yearly seasonality and a near-linear trend (last-year's-value plus a
  similar yearly drift is almost the right answer).
- The hybrid model becomes more valuable when (a) the residuals contain
  exploitable structure beyond simple seasonality, or (b) you have
  exogenous regressors (volcanic activity, El Niño indices, …) that the
  classical models can't easily use.

The lesson: think about your data's structure *first*, then pick a
modelling approach that respects it.


## Exercises

**Exercise 6.1**: *Confirm the trend trap.*
Fit XGBoost directly on the **raw** CO2 series (no detrending), using
the same lag + calendar features. Run the iterative forecast and add it
to the scoreboard. How much worse is it than the hybrid model, and
exactly where on the forecast plot does it fail?

**Exercise 6.2**: *Direct vs iterative forecasting.*
Implement a **direct** multi-step forecast on the residuals: train a
separate XGBoost model for each horizon $h \in \{1, \ldots, 52\}$ using
only lags $\geq h$ (which are guaranteed to be available $h$ steps
ahead). Compare its RMSE against the iterative hybrid above.

**Exercise 6.3 (stretch)**: *Hyperparameter CV with `TimeSeriesSplit`.*
Use `sklearn.model_selection.TimeSeriesSplit(n_splits=4)` on the
training **residuals** to grid-search

- `n_estimators ∈ {100, 300, 500}`
- `max_depth ∈ {3, 4, 5}`

Pick the lowest-average-RMSE combination and re-fit. Did test-set RMSE
improve over the default?

*Note: do NOT use plain `KFold`, random shuffling would leak future
information into the training fold.*


---

*Chen, T. & Guestrin, C. (2016), XGBoost: A Scalable Tree Boosting
System. KDD '16. <https://arxiv.org/abs/1603.02754>*
